# Clustering Validation Notebook
This notebook computes a comprehensive set of validation metrics for HDBSCAN clustering of Philadelphia business centroids.
Metrics include internal, HDBSCAN-specific, fuzzy, external, and spatial validation methods.

In [7]:
import os
import numpy as np
import geopandas as gpd
import hdbscan
import torch
from torchmetrics.clustering import DunnIndex
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    rand_score,
    fowlkes_mallows_score,
    normalized_mutual_info_score
)
from sklearn.neighbors import NearestNeighbors
from s_dbw import S_Dbw
from scipy.spatial.distance import pdist, squareform, cdist
import skfuzzy as fuzz  
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local
from esda.geary import Geary

In [8]:
# Load data
biz_gdf = gpd.read_file('../../precomputed/businesses.geojson')
biz_gdf = biz_gdf.to_crs(epsg=32618)
coords = np.vstack([biz_gdf.geometry.y, biz_gdf.geometry.x]).T
features = biz_gdf[['sentiment_score', 'subjectivity_score', 'stars_rev']].copy()
features['business_count'] = 1

In [9]:
# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(features)

In [15]:
# Fit HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=12,
                            metric='euclidean', cluster_selection_method='leaf',
                            gen_min_span_tree=True, approx_min_span_tree=False)
labels = clusterer.fit_predict(X)
biz_gdf["cluster_id"] = labels

c:\Users\alche\OneDrive\Documents\GitHub\AllenCheung0213.github.io\CS554_NLP_Final_Project\newenv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\alche\OneDrive\Documents\GitHub\AllenCheung0213.github.io\CS554_NLP_Final_Project\newenv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [16]:
# Filter noise
mask = labels != -1
X_in = X[mask]
labels_in = labels[mask]

In [17]:
# Internal metrics
if len(np.unique(labels_in)) > 1:
    sil = silhouette_score(X_in, labels_in)
    db  = davies_bouldin_score(X_in, labels_in)
    ch  = calinski_harabasz_score(X_in, labels_in)
    print(f"Silhouette Coefficient: {sil:.3f}")
    print(f"Davies–Bouldin Index:    {db:.3f}")
    print(f"Calinski–Harabasz Index: {ch:.1f}")
else:
    print("Not enough clusters (after noise removal) to compute internal metrics.")

Silhouette Coefficient: 0.465
Davies–Bouldin Index:    0.512
Calinski–Harabasz Index: 2735.3


In [18]:
dunn_metric = DunnIndex()
data_tensor = torch.tensor(X_in, dtype=torch.float32)
labels_tensor = torch.tensor(labels_in, dtype=torch.int32)
dunn_score = dunn_metric(data_tensor, labels_tensor)
print(f"Dunn Index: {dunn_score.item():.3f}")

Dunn Index: 0.178


In [19]:
print(f"Relative Validity: {clusterer.relative_validity_:.3f}")
print(f"Cluster Persistence: {clusterer.cluster_persistence_!r}")

Relative Validity: 0.012
Cluster Persistence: array([0.00244322, 0.01466509, 0.03786088, 0.01036827, 0.0235624 ,
       0.02413927, 0.01421318, 0.03907232, 0.1227063 , 0.01976223,
       0.10781871])


In [ ]:
# Xie-Beni Index
def xie_beni_index(X: np.ndarray, labels: np.ndarray) -> float:
    unique_labels = np.unique(labels)
    k = unique_labels.size
    N = X.shape[0]
    centroids = np.array([
        X[labels == lbl].mean(axis=0)
        for lbl in unique_labels
    ])
    wgss = 0.0
    for lbl, ctr in zip(unique_labels, centroids):
        pts = X[labels == lbl]
        wgss += np.sum((pts - ctr)**2)
    compactness = wgss / N
    if k > 1:
        dists = cdist(centroids, centroids, metric='euclidean')
        np.fill_diagonal(dists, np.inf)
        min_centroid_sq = np.min(dists)**2
    else:
        return np.inf
    xb_value = compactness / min_centroid_sq
    return xb_value

xb = xie_beni_index(X_in, labels_in)
print(f"Xie–Beni Index: {xb}")

Xie–Beni Index: 2.2789953329470922


In [25]:
# S_Dbw Index
sd_bw_score = S_Dbw(X_in, labels_in, centers_id=None,
                    method='Tong', alg_noise='bind',
                    centr='mean', nearest_centr=True,
                    metric='euclidean')
print(f"S-Dbw Index: {sd_bw_score:.3f}")

S-Dbw Index: 0.095


In [26]:
# C-Index
def c_index(X, labels):
    D = squareform(pdist(X, 'euclidean'))
    intra = [D[i,j] for i in range(len(labels))
                     for j in range(i)
                     if labels[i]==labels[j]]
    m = len(intra)
    S, all_d = sum(intra), np.sort(D.flatten())
    return (S - all_d[:m].sum()) / (all_d[-m:].sum() - all_d[:m].sum())
ci = c_index(X, labels)
print("C-Index:", ci)

C-Index: 0.46989867854597134


In [28]:
# Connectivity Index
def connectivity_index(X, labels, n_neighbors=8):
    nbrs = NearestNeighbors(n_neighbors=n_neighbors+1).fit(X)
    _, indices = nbrs.kneighbors(X)
    conn = 0
    for i in range(X.shape[0]):
        for j in indices[i, 1:]:
            if labels[i] != labels[j]:
                conn += 1
    return conn

conn = connectivity_index(X, labels, n_neighbors=8)
print(f"Connectivity Index: {conn}")

Connectivity Index: 3765


In [29]:
# Fuzzy Clustering Metrics
data_t = X_in.T
n_clusters = len(np.unique(labels_in))
cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    data_t,
    c=n_clusters,
    m=2.0,
    error=0.005,
    maxiter=1000,
    init=None,
    seed=42
)

pc = np.sum(u**2) / u.shape[1]
pe = -np.sum(u * np.log(u + 1e-12))

print(f"Partition Coefficient: {pc:.3f}")
print(f"Partition Entropy:     {pe:.3f}")

Partition Coefficient: 0.670
Partition Entropy:     565.316


In [30]:
# Spatial Autocorrelation
w = KNN.from_dataframe(biz_gdf, geom_col="geometry", k=8)
w.transform = "R"
moran_global = Moran(biz_gdf["cluster_id"], w)
geary_global = Geary(biz_gdf["cluster_id"], w)
lisa = Moran_Local(biz_gdf["cluster_id"], w)
print(f"Global Moran’s I: {moran_global.I:.3f}")
print(f"Geary’s C: {geary_global.C:.3f}")
print(f"Local Moran’s I p-values (first 5): {lisa.p_sim[:5]}")

c:\Users\alche\OneDrive\Documents\GitHub\AllenCheung0213.github.io\CS554_NLP_Final_Project\newenv\Lib\site-packages\libpysal\weights\distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


Global Moran’s I: 0.010
Geary’s C: 0.989
Local Moran’s I p-values (first 5): [0.483 0.483 0.24  0.483 0.001]
